In [2]:
import pandas as pd
df = pd.read_csv('../data/test.csv')
df.head()

,text,label
0,im feeling rather rotten so im not very ambiti...,0
1,im updating my blog because i feel shitty,0
2,i never make her separate from me because i do...,0
3,i left with my bouquet of red and yellow tulip...,1
4,i was feeling a little vain when i did this one,0


In [3]:
df['label'].value_counts().sort_index()

label
0    581
1    695
2    159
3    275
4    224
5     66
Name: count, dtype: int64

In [4]:
for label in sorted(df['label'].unique()):
    print(f"label {label}:")
    print(df[df['label']==label]['text'].iloc[0])
    print()

label 0:
im feeling rather rotten so im not very ambitious right now

label 1:
i left with my bouquet of red and yellow tulips under my arm feeling slightly more optimistic than when i arrived

label 2:
i find myself in the odd position of feeling supportive of

label 3:
i felt anger when at the end of a telephone call

label 4:
i cant walk into a shop anywhere where i do not feel uncomfortable

label 5:
i feel a little stunned but can t imagine what the folks who were working in the studio up until this morning are feeling



In [5]:
label_to_emotion = {
    0: 'sadness',
    1: 'joy',
    2: 'love',
    3: 'anger',
    4: 'fear',
    5: 'surprise'
}

In [7]:
import sys
!{sys.executable} -m pip install nltk

  Using cached nltk-3.10.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached regex-2026.7.19-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tqdm-4.70.0-py3-none-any.whl.metadata (57 kB)
Using cached nltk-3.10.3-py3-none-any.whl (1.8 MB)
Using cached regex-2026.7.19-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (801 kB)
Using cached click-8.5.0-py3-none-any.whl (125 kB)
Using cached tqdm-4.70.0-py3-none-any.whl (80 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [nltk]━━━━━━ 3/4 [nltk]]


In [8]:
import nltk
nltk.download("stopwords")

/home/biplob/venvs/data/lib/python3.14/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/biplob/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
[nltk_data] Downloading package stopwords to /home/biplob/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [20]:
import re
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
stop_words.discard('not')

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]','',text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)
    

In [21]:
sample = df['text'].iloc[0]
print("Before:",sample)
print("After:",clean_text(sample))

Before: im feeling rather rotten so im not very ambitious right now
After: im feeling rather rotten im not ambitious right


In [22]:
df['clean_text']=df['text'].apply(clean_text)
df.head()

,text,label,clean_text
0,im feeling rather rotten so im not very ambiti...,0,im feeling rather rotten im not ambitious right
1,im updating my blog because i feel shitty,0,im updating blog feel shitty
2,i never make her separate from me because i do...,0,never make separate ever want feel like ashamed
3,i left with my bouquet of red and yellow tulip...,1,left bouquet red yellow tulips arm feeling sli...
4,i was feeling a little vain when i did this one,0,feeling little vain one


In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorize =  TfidfVectorizer()
X= vectorize.fit_transform(df['clean_text'])

print(X.shape)

(2000, 4646)


In [24]:
from sklearn.model_selection import train_test_split

Y = df['label']
X_train,X_text,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

print("Train size:", X_train.shape)
print("Test size:",X_text.shape)

Train size: (1600, 4646)
Test size: (400, 4646)


In [25]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train,Y_train)

accuracy = model.score(X_text,Y_test)
print("Accuracy:",accuracy)

Accuracy: 0.61


In [26]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train,Y_train)

dt_accuracy=dt_model.score(X_text,Y_test)
print("Decision tree Accuracy:",dt_accuracy)

Decision tree Accuracy: 0.6125


In [27]:
from sklearn.svm import SVC

svm_model = SVC()
svm_model.fit(X_train,Y_train)

svm_accuracy= svm_model.score(X_text,Y_test)
print("Support Vector Accuracy:",svm_accuracy)

Support Vector Accuracy: 0.565


In [28]:
import pickle
with open('../models/my_model.pkl','wb') as f:
    pickle.dump(model,f)
with open('../models/my_vectorizer.pkl','wb') as f:
    pickle.dump(vectorize,f)
with open('../models/label_mapping.pkl','wb') as f:
    pickle.dump(label_to_emotion,f)

In [29]:
import pickle

with open('../models/my_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

with open('../models/my_vectorizer.pkl', 'rb') as f:
    loaded_vectorizer = pickle.load(f)

with open('../models/label_mapping.pkl', 'rb') as f:
    loaded_mapping = pickle.load(f)

def predict_emotion(text):
    cleaned = clean_text(text)
    vector = loaded_vectorizer.transform([cleaned])
    prediction = loaded_model.predict(vector)[0]
    return loaded_mapping[prediction]

print(predict_emotion("I am so happy today, everything is amazing"))
print(predict_emotion("I feel scared and anxious about tomorrow"))

joy
fear
